# Comparación de métodos de clustering
K-Means, clustering jerárquico, DBSCAN y mezclas gaussianas (GMM).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.datasets import make_moons, make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

X, _ = make_blobs(n_samples=500, cluster_std=2.5, random_state=42)
X = StandardScaler().fit_transform(X)

plt.scatter(X[:, 0], X[:, 1], s=15)
plt.title("Datos de ejemplo")
plt.show()

## 1. K-Means
Agrupa observaciones alrededor de centroides. Cambiamos el número de clusters.

In [ ]:
kmeans_2.cluster_centers_

In [ ]:
kmeans_2 = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans_3 = KMeans(n_clusters=3, random_state=42, n_init=10)

kmeans_2_labels = kmeans_2.fit_predict(X)
kmeans_3_labels = kmeans_3.fit_predict(X)

fig, ax = plt.subplots(1, 2, figsize=(10, 6))
for a, etiquetas, titulo in zip(ax, [kmeans_2_labels, kmeans_3_labels], ["k = 2", "k = 3"]):
    a.scatter(X[:, 0], X[:, 1], c=etiquetas, cmap="tab10", s=15)
    a.set_title(titulo)

ax[0].scatter(kmeans_2.cluster_centers_[:,0], kmeans_2.cluster_centers_[:,1], c="red", marker="x")
ax[1].scatter(kmeans_3.cluster_centers_[:,0], kmeans_3.cluster_centers_[:,1], c="red", marker="x")
plt.tight_layout()
plt.show()

## 2. Clustering jerárquico
Construye grupos a partir de la distancia entre observaciones. Comparamos dos tipos de enlace.

In [ ]:
jerarquico_ward = AgglomerativeClustering(n_clusters=2, linkage="ward").fit_predict(X)
jerarquico_single = AgglomerativeClustering(n_clusters=3, linkage="single").fit_predict(X)

fig, ax = plt.subplots(1, 2, figsize=(10, 6))
for a, etiquetas, titulo in zip(ax, [jerarquico_ward, jerarquico_single], ["Enlace: ward", "Enlace: single"]):
    a.scatter(X[:, 0], X[:, 1], c=etiquetas, cmap="tab10", s=15)
    a.set_title(titulo)
plt.tight_layout()
plt.show()

## 3. DBSCAN
Detecta regiones densas y puede identificar ruido (`-1`). Cambiamos `eps`.

In [ ]:
dbscan_015 = DBSCAN(eps=0.15, min_samples=10).fit_predict(X)
dbscan_025 = DBSCAN(eps=0.25, min_samples=10).fit_predict(X)

fig, ax = plt.subplots(1, 2, figsize=(10, 6))
for a, etiquetas, titulo in zip(ax, [dbscan_015, dbscan_025], ["eps = 0.15", "eps = 0.25"]):
    a.scatter(X[:, 0], X[:, 1], c=etiquetas, cmap="tab10", s=15)
    a.set_title(titulo)
plt.tight_layout()
plt.show()

## 4. Gaussian Mixture Models (GMM)
Modela los datos como una mezcla de distribuciones gaussianas. Cambiamos el número de componentes.

In [ ]:
gmm_2 = GaussianMixture(n_components=2, random_state=42).fit_predict(X)
gmm_3 = GaussianMixture(n_components=3, random_state=42).fit_predict(X)
gmm_4 = GaussianMixture(n_components=4, random_state=42).fit_predict(X)

fig, ax = plt.subplots(1, 3, figsize=(12, 6))
for a, etiquetas, titulo in zip(ax, [gmm_2, gmm_3, gmm_4],
 ["Componentes = 2", "Componentes = 3", "Componentes = 4"]):
    a.scatter(X[:, 0], X[:, 1], c=etiquetas, cmap="tab10", s=15)
    a.set_title(titulo)
plt.tight_layout()
plt.show()

## 5. Comparación con el coeficiente de silueta
Valores cercanos a **1** indican grupos más separados; valores cercanos a **0**, grupos solapados. Se comparan las ocho configuraciones. Para DBSCAN se excluye el ruido (`-1`) del cálculo.

In [ ]:
resultados = {
    "K-Means (k=2)": kmeans_2_labels,
    "K-Means (k=3)": kmeans_3_labels,
    "Jerárquico (ward)": jerarquico_ward,
    "Jerárquico (single)": jerarquico_single,
    "DBSCAN (eps=0.15)": dbscan_015,
    "DBSCAN (eps=0.25)": dbscan_025,
    "GMM (2 componentes)": gmm_2,
    "GMM (3 componentes)": gmm_3,
    "GMM (4 componentes)": gmm_4
}

filas = []
for nombre, etiquetas in resultados.items():
    mascara = etiquetas != -1  # Excluir ruido de DBSCAN
    grupos = np.unique(etiquetas[mascara])
    silueta = (silhouette_score(X[mascara], etiquetas[mascara])
                if 1 < len(grupos) < mascara.sum() else np.nan)
    filas.append([nombre, silueta])

comparacion = pd.DataFrame(filas, columns=["Configuración", "Silueta"])
print(comparacion.round(3).to_string(index=False))

comparacion.plot.bar(x="Configuración", y="Silueta", legend=False, figsize=(9, 4))
plt.ylabel("Coeficiente de silueta")
plt.title("Comparación de configuraciones")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

**Nota:** una silueta mayor no siempre significa que el agrupamiento sea más adecuado: por ejemplo, DBSCAN puede reconocer formas no convexas que la silueta penaliza.